
# Do LLMs know the difference between a pet chicken and a roast chicken?

## Word sense disambiguation in computational models and humans


In human language, words do not always have a fixed meaning. The most striking example is homonymous words: words that have the same form, but very different meanings. For instance, the word "bank", which has a different meaning in the context "I went to the bank to get some money" and "At the river bank, I met my old friend". Polysemous words are words that have different -- yet related -- meanings: for example, "chicken" is the same 'entity' in "My pet chicken is lovely" and "I am having roast chicken for dinner", but has very different meanings in these two contexts. In general, context can modulate almost any word's meaning. This poses a challenge in computational linguistics, as we need to find a way to differentiate among different meanings like humans do. Much research, resources, and models have been put forward to help with this challenge.

In this assignment, you are going to focus on [Trott and Bergen's (2021)](https://aclanthology.org/2021.acl-long.550/) RAW-C dataset: you are going to conduct a number of explorations with this dataset and partially replicate their research by the end of the assignment. In short, the authors explore how good LLMs are at capturing same/different meanings of words across contexts by comparing it to human judgements. To better understand the idea and the research, start by reading the paper.

This assignment entails a series of (interconnected) tasks (altogether worth 95 points):

* **Task 1**. Compute contextual word embeddings at different layers from Trott & Bergen's dataset. Here, each word is found in 4 sentences: 2 with one meaning, 2 with another meaning.
* **Task 2**. Compute sense embeddings for words in Trott & Bergen's dataset using WordNet, so you have an embedding for each definition of the word.
* **Task 3**. Compute the similarity between the contextual word embeddings of the homonyms at different layers and their sense embeddings; explore the relationship between homonyms and dominant senses quantitatively and qualitatively
* **Task 4**. Replicate part of Trott & Bergen's work by computing similarities across sentences with same/different meanings at the different layers and correlate with human similarities; visualise the results and reflect on them

In order to better understand the assignment, we recommend going through it all before starting so that it is clear how each part is connected to the next (which will help you make decisions about data structures, for instance).

# Task 1: Compute contextual word embeddings for homonyms [20 points]

## Task 1.1: read, explore and extract the necessary data [5 points]

First, you will have to (fork and) clone the github repository that stores the data you'll need. This can be found here: https://github.com/sashakenjeeva/raw-c . The repo also includes a README with a description of the original files in the repository, as well as some notes relevant for this assignment specifically.

In [ ]:
# your code here (you can use as many cells as necessary/you prefer)

Make sure you mount the drive now so that you have access to the folder (think about setting the working directory in a way that is convenient).

In [ ]:
# mount the drive here

Now, you will have to read the data and organise it in a structure that works for the next parts of the assignment.

Read and explore the dataframe to see its structure (print part of it). What we need from it are the homonyms (in the form that they appear in the sentence -- the lexeme -- and in their regular form -- the lemma) and their corresponding sentences with different meanings (M1_a and M1_b have same meaning; M2_a, M2_b have same meaning). We only will need the stimuli that are in the final RAW-C dataset, as this is what we'll replicate at the end.

You can decide which data structure to use, but make sure that all these pieces of information are there (the word, the string, the meaning id, and the corresponding sentences) and easy to retrieve. Show your data at the end, as well as how many stimuli you end up with.

In [1]:
import pandas as pd

In [2]:
# read the data here
# remember to print the data structure you produce and the number of stimuli it contains!

df_rawc = pd.read_csv("data/processed/raw-c.csv")


In [3]:
df_rawc.head()

,word,sentence1,sentence2,same,ambiguity_type,disambiguating_word1,disambiguating_word2,version,Class,mean_relatedness,median_relatedness,diff,count,sd_relatedness,distance_bert,distance_elmo,se_relatedness,v1,v2,string
0,act,It was a desperate act.,It was a magic act.,False,Polysemy,desperate,magic,M1_a_M2_a,N,2.181818,2.0,0.181818,11,1.328020,0.204110,0.034093,0.400413,M1_a,M2_a,act
1,act,It was a desperate act.,It was a comedic act.,False,Polysemy,desperate,comedic,M1_a_M2_b,N,2.000000,2.0,0.000000,7,1.290994,0.215616,0.045927,0.487950,M1_a,M2_b,act
2,act,It was a humane act.,It was a magic act.,False,Polysemy,humane,magic,M1_b_M2_a,N,2.818182,3.0,0.181818,11,0.981650,0.191488,0.042351,0.295979,M1_b,M2_a,act
3,act,It was a humane act.,It was a comedic act.,False,Polysemy,humane,comedic,M1_b_M2_b,N,2.809524,3.0,0.190476,21,0.928388,0.225272,0.057707,0.202591,M1_b,M2_b,act
4,act,It was a desperate act.,It was a humane act.,True,Polysemy,desperate,humane,M1_a_M1_b,N,3.900000,4.0,0.100000,10,0.316228,0.167990,0.041440,0.100000,M1_a,M1_b,act


In [4]:
print(df_rawc.shape)
print(df_rawc.dtypes)
df_rawc.info()


(672, 20)
word                        str
sentence1                   str
sentence2                   str
same                       bool
ambiguity_type              str
disambiguating_word1        str
disambiguating_word2        str
version                     str
Class                       str
mean_relatedness        float64
median_relatedness      float64
diff                    float64
count                     int64
sd_relatedness          float64
distance_bert           float64
distance_elmo           float64
se_relatedness          float64
v1                          str
v2                          str
string                      str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   word                  672 non-null    str    
 1   sentence1             672 non-null    str    
 2   sentence2             672 non-nul

In [16]:
df = df_rawc[['word', 'string', 'sentence1', 'sentence2', 'v1', 'v2']]
df = df.rename(columns = {'word': 'lemma', 'string': 'word'})

In [19]:
df

,lemma,word,sentence1,sentence2,v1,v2
0,act,act,It was a desperate act.,It was a magic act.,M1_a,M2_a
1,act,act,It was a desperate act.,It was a comedic act.,M1_a,M2_b
2,act,act,It was a humane act.,It was a magic act.,M1_b,M2_a
3,act,act,It was a humane act.,It was a comedic act.,M1_b,M2_b
4,act,act,It was a desperate act.,It was a humane act.,M1_a,M1_b
...,...,...,...,...,...,...
667,yard,yards,It was five yards.,They were cluttered yards.,M1_a,M2_b
668,yard,yards,It was ten yards.,They were big yards.,M1_b,M2_a
669,yard,yards,It was ten yards.,They were cluttered yards.,M1_b,M2_b
670,yard,yards,It was five yards.,It was ten yards.,M1_a,M1_b


In [17]:
df_sentences = pd.concat([
    df[['lemma', 'word', 'sentence1', 'v1']].rename(columns={'sentence1': 'sentence', 'v1': 'v'}),
    df[['lemma', 'word', 'sentence2', 'v2']].rename(columns={'sentence2': 'sentence', 'v2': 'v'}),
]).drop_duplicates(subset=['lemma', 'word', 'sentence']).reset_index(drop=True)

In [18]:
df_sentences

,lemma,word,sentence,v
0,act,act,It was a desperate act.,M1_a
1,act,act,It was a humane act.,M1_b
2,act,act,It was a magic act.,M2_a
3,appeal,appeal,He had a universal appeal.,M1_a
4,appeal,appeal,He had an undefinable appeal.,M1_b
...,...,...,...,...
443,title,title,She liked the professional title.,M2_b
444,toast,toasted,They toasted the bride.,M2_b
445,toll,toll,It was a bell toll.,M2_b
446,track,tracks,She saw the snowshoe tracks.,M2_b


## Task 1.2: Compute the contextualised word embeddings [15 points]


Now that you have the homonyms and their corresponding sentences, we will need to compute word embeddings for each of them. For this we will use the BERT base model, in its uncased version.

That is, for each homonym, you will have to compute four embeddings: one for the homonym in M1_a, one in M1_b, one in M2_a, one in M2_b. However, we also want to look into different layers of the BERT model to see which one captures the homonym's meaning best: you want to calculate embeddings at the static layer and at layers 4, 8, 12.

We will use the package psycho-embeddings (you will use it in class), which allows us to specify which target words we want to obtain the embeddings of, in which sentences, and at which layers, among other things. Make sure to read the documentation of the package so that you know the meaning of the arguments and which ones will come useful to you.

First of all, install the psycho-embeddings package below.

In [ ]:
# install the psycho-embeddings package here

/home/mateu/workspace/comp_ling_assignment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Now, import the relevant module/function from psycho-embeddings and load the required BERT model.

In [7]:
# your code here
from psycho_embeddings import ContextualizedEmbedder

Now, test that everything works correctly by computing an embedding for the word "assignment" in the sentence "I am having so much fun with this assignment!", at static layer and layers 4, 8 and 12 (hint: think of tokenisation and how the embedder deals with that).

In [8]:
# your code here

model = ContextualizedEmbedder("bert-base-cased", max_length=128)

loading configuration file config.json from cache at /home/mateu/.cache/huggingface/hub/models--bert-base-cased/snapshots/cd5ef92a9fb2f889e972770a36d4ed042daf221e/config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_hidden_states": true,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.5.4",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}

loading weights file model.safetensors 

In [10]:
embeddings = model.embed(
    words=["assignment"],
    target_texts=["I am having so much fun with this assignment!"],
    layers_id=[4, 8, 12],
    batch_size=8,
    return_static=True,
)

Parameter 'function'=<function ContextualizedEmbedder.embed.<locals>.tokenizer_function at 0x76585817b560> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.
  0%|          | 0/1 [00:00<?, ?it/s]/home/mateu/workspace/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 1/1 [00:00<00:00,  2.74it/s]


In [20]:
print(embeddings.keys())
print(len(embeddings))

dict_keys([4, 8, 12, -1])
4


The next step is to calculate embeddings for the homonyms and their sentences that we got from the RAW-C dataset.

Make sure that your final output includes the word, the meaning id (M1_a, etc), the corresponding sentence and the embeddings at static layer and layers 4, 8, 12. You should maximally optimise this process by calculating in batches (again, check psycho-embeddings documentation), but keep in mind this might still take a while. First test your pipeline with a small number of inputs, and only run the full scale embedding extraction once you're positive the code works as expected.

When done, save the output in [pickle](https://docs.python.org/3/library/pickle.html) format (this is similar to json, but it can also handle np.arrays), so that you can easily load it later when needed and do not have to run it again. After pickle dumping (that's the word for saving it in pickle format), print it so that you are sure everything was saved correctly.

Then, check that your final data includes everything that you need by checking the entry "bank" and print the data pertaining to "bank".

In [21]:
# your code here

def get_embeddings(row):
    result = model.embed(
        words=[row["word"]],
        target_texts=[row["sentence"]],
        layers_id=[4, 8, 12],
        return_static=True,
    )
    return {
        f"layer_{k}" if k != -1 else "static": v[0] 
        for k, v in result.items()
    }



In [22]:
embedding_cols = df_sentences.apply(get_embeddings, axis = 1, result_type="expand")
df_sentences = pd.concat([df_sentences, embedding_cols], axis = 1)

  0%|          | 0/1 [00:00<?, ?it/s]/home/mateu/workspace/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/1 [00:00<?, ?it/s]/home/mateu/workspace/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/1 [00:00<?, ?it/s]/home/mateu/workspace/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/1 [00:00<?, ?it/s]/home/mateu/workspace/comp_ling_assignment/.venv/lib/python3.12/site-

In [23]:
df_sentences

,lemma,word,sentence,v,layer_4,layer_8,layer_12,static
0,act,act,It was a desperate act.,M1_a,"[-0.058270857, -0.27012166, 0.4256111, -0.1391...","[-0.16379957, 0.28225708, 0.25096044, 0.133226...","[-0.07384763, 0.29591572, 0.20930922, 0.215898...","[-0.0055097593, -0.029294828, -0.013608456, -0..."
1,act,act,It was a humane act.,M1_b,"[0.08830817, -0.4468078, 0.38992456, -0.443280...","[-0.20170821, -0.2736744, 0.08747089, -0.19021...","[0.17367648, -0.016312934, 0.08041777, 0.25971...","[-0.0055097593, -0.029294828, -0.013608456, -0..."
2,act,act,It was a magic act.,M2_a,"[-0.27173337, -0.39731818, 0.7297983, -0.57784...","[-0.13248068, -0.059307016, 0.36231703, -0.200...","[0.06568524, 0.17123552, 0.0786918, 0.17533155...","[-0.0055097593, -0.029294828, -0.013608456, -0..."
3,appeal,appeal,He had a universal appeal.,M1_a,"[0.9152113, 1.000226, -1.0877796, 1.219724, 0....","[1.0407547, 1.0634527, -0.084837765, 0.6938994...","[0.39856434, 0.50855243, 0.10799514, 0.3002188...","[0.052237112, 0.06373871, -0.097832315, 0.0396..."
4,appeal,appeal,He had an undefinable appeal.,M1_b,"[0.37407812, 0.4802913, -0.7315045, 0.95309293...","[0.7130865, 0.8216304, -0.084937304, 0.9353232...","[0.41320452, 0.41399145, 0.07545343, 0.3192787...","[0.052237112, 0.06373871, -0.097832315, 0.0396..."
...,...,...,...,...,...,...,...,...
443,title,title,She liked the professional title.,M2_b,"[-0.31480634, 0.32864818, 0.19436222, 0.727830...","[-0.36328006, 0.67257, 0.28871232, 0.8307625, ...","[0.08188132, 0.005937785, 0.42133906, 0.253184...","[-0.040719163, -0.0012473409, -0.02736756, 0.0..."
444,toast,toasted,They toasted the bride.,M2_b,"[-0.11858702, -0.014792069, -0.30728698, 1.191...","[-0.46605524, 0.24366137, -0.18897048, 0.39927...","[-0.25262582, -0.018994898, 0.2654907, -0.1089...","[-0.024888199, 0.043743096, -0.042333048, 0.03..."
445,toll,toll,It was a bell toll.,M2_b,"[-1.2667142, 0.29585883, 0.3312982, 0.2920356,...","[-1.3029038, -0.5709333, -0.59905964, -0.05065...","[-0.33465624, -0.02796751, 0.25729203, 0.26721...","[-0.059510894, -0.007590637, 0.07930978, -0.06..."
446,track,tracks,She saw the snowshoe tracks.,M2_b,"[0.111913115, 0.18632863, -0.13256183, -0.4852...","[0.28136355, 0.053297512, 0.26244485, 0.534611...","[0.15402, -0.15668586, 0.20108183, -0.15116693...","[-0.062270977, -0.02131133, 0.06818508, -0.016..."


In [24]:
df.to_pickle("data/processed/with_embs.pkl")

# Task 2: Compute sense embeddings for the homonym dataset using WordNet [20 points]

Your next task is to fetch the definitions (glosses) of the homonyms, and compute an embedding for each gloss (each gloss is associated with a specific sense). We do that so we can later see whether the contextualised embeddings computed above represent the meaning of the homonym in context well (by comparing it to the sense embeddings). Figure 18.9 in [Jurafsky's and Martin's (2021) chapter 18](https://web.stanford.edu/~jurafsky/slp3/old_sep21/18.pdf) graphically illustrates this idea. Use this chapter for this part of the assignment, as it will come useful for you both theoretically and practically.

## Task 2.1: Fetch senses and glosses for a word [5 points]

First of all, you will have to figure out how [WordNet](https://www.nltk.org/howto/wordnet.html) works within the nltk package (hint: pay attention to what a synset is).

Install and import all the necessary components and define a function to extract the glosses of a word and create a dictionary with senses and glosses.

Then use the word "bat" to test that everything is working correctly: i.e., for "bat", you should be able to get its senses and the gloss for each of the sense (you will see that synsets might contain related words, but you only need the senses that contain the word of interest or derivates thereof; this should be specified in the function). Print the output for "bat".


In [ ]:
# your code here

## Task 2.2: Function to compute sense embeddings [10 points]

Now that you have a function to extract senses and glosses for a given word, write a function that takes a word and computes embeddings for each of the senses following the method explained in Jurafsky's and Martin's chapter. In this case, no need to calculate at different layers: you should use the last layer only. You should maximally optimise this function like before.

The output should include the sense, the gloss, and the embedding. Print the function's output when using the word "bank".


In [ ]:
# your code here

## Task 2.3: Compute sense embeddings for the RAW-C stimuli [5 points]

Now, use the function you defined above to compute sense embeddings for the RAW-C stimuli and pickle dump it too.

As above, the information that should be there for each word is: the sense, the gloss, the embedding at the last layer. Again, you can think of which structure to use best, but keep in mind that we will have to compare these to the CWE calculated in task 1, so it is good to think of a similar structure that is easily comparable.

Make sure that the number of stimuli matches the number of stimuli in the final RAW-C dataset.

In [ ]:
# your code here

# Task 3: Compute and explore similarity between homonym CWEs and sense embeddings [35 points]

You now have the homonym CWEs computed in task 1, and the sense embeddings computed in task 2. The next step is to calculate cosine similarities between each CWE for each homonym (at the selected layer!) and each sense embedding for that homonym.

For instance, say for the word "bat" with meaning M1_a, you have its CWE at the static layer and at layers 4, 8, 12 and 7 senses: here, you will end up with 16 cosine similarities (take each CWE and compute its similarity to each of the sense embeddings). We then want to see which sense meaning is the closest to each CWE, and do some qualitative explorations with that.

## Task 3.1: Compute the cosine similarity between all the CWEs and the sense embeddings [8 points]

This task is not trivial with regards to how much information you have and how to structure the data (this is why it's also important to think of data structures in the earlier parts of the assignment), so take some time to think how to best breakdown this task. Test each step/function if you have multiple. Pickle dump your final output so that it is easily retrievable for later. At the end, print an example of the entry "bank".

For cosine similarity, the cdist function from scipy.spatial.distance seems the most efficient, but you are free to use any of your liking (hint: pay attention to the shape of your embeddings and to similarity vs distance. You will need the similarity).

In [ ]:
# your code here

## Task 3.2: Quantitative and qualitative explorations the relationship between homonym embeddings and dominant senses

Now, we can look into how the CWEs in different meanings and layers relate to the different senses of a homonym. We'll focus on the dominant sense in WordNet, see below for more details. This section includes both code blocks and reflection questions.

### Dominant senses in WordNet and top senses across layers (focus on static layer) [8 points]

Embeddings at the static layer do not take into account context, so intuitively they should capture the 'average' meaning, maybe the most common/dominant. We can test this by looking at the most similar sense and seeing if that matches that most common/dominant sense in the synset.

Keep in mind that synsets mark more common/dominant senses with numbering: so n.01 will be the most common noun; v.01 the most common verb, etc. If that is not available, the most common meaning will be the next number (e.g., n.02). You have to take that into account when you extract the top sense, so first extract information about which are the most dominant senses for each word across all the parts of speech: for example, "bat" might have as its two most common senses bat.n.01 and bat.v.02 (because v.01 might not be available; this is just an example). Some words might only have one part of speech in their synset, some more. Print your results.

In [ ]:
# your code here

Then, extract the top similarity of homonyms to the senses at all the layers you have available. While we are interested in the static layer for checking dominant senses, it is also interesting to look into other layers to see whether adding context will refine the captured meaning.


In [ ]:
# your code here

Let's check an example from our results.

Out of all the similarities of 'bank' to all its senses at all the layers, which one is the highest? Print your results for that entry and reflect below.

In [ ]:
# your code here

### Does the static layer capture the most dominant meaning, according to WordNet (and according to you)? [2 point]

%your answer here

### Across other layers and meanings, which layer seems to capture the meaning of bank across meanings best, and why do you make this conclusion? [2 points]

%your answer here

### Checking matches and mismatches with the dominant sense [5 points]

Now, let's quantitatively check if the static layer actually captures the most dominant sense (any POS). You should end up with two data structures: matches (when the most similar sense is one of the dominant senses) and mismatches (when the most similar sense is not one of the dominant sense). Do that also for the other layers to compare. Print the percentage of matches and mismatches per layer.



In [ ]:
# your code here

Now, print the matches and mismatches for the static layer only.

In [ ]:
# your code here

### Do BERT's static embeddings capture the most dominant sense in WordNet? [2 point]

%your answer here

### Do the percentages of matches and mismatches throughout the layers make sense to you or is it different than what you expected? [2 points]

%your answer here

### For the **static layer**, are there any words that seem to particularly deviate from the dominant meaning? If so, which and why could that be? [3 points]

%your answer here

### Do you think the corpus on which BERT is trained might reflect different meaning dominance than for WordNet's senses? If so/not, why? [3 points]

%your answer here

# Task 4: Partially replicate Trott & Bergen's experiment [20 points]

Now comes the time to partially replicate the RAW-C experiment, by seeing whether different layers of BERT capture meanings more or less similarly to humans. At the end you will have to wrap up with a brief comment on which layer seems to capture meanings best and how that connects to explorations in the previous section.

## Task 4.1: Create a dataframe with cosine similarities between sentences at different layers [7 points]

You should now use the embeddings at the different layers that you computed to calculate similarities between each context: M1a, M1b, M2a, M2b. You will have to have all combinations, so for each string in the RAW-C dataframe, you'll have: M1a vs M1b, M1a vs M2a, M1a vs M2b, M1b vs M2a, M1b vs M2b, M2a vs M2b.

Bear in mind that your final dataframe should include: the word, the string as it appears in the sentence, cosine similarity at layers 4, layer 8, layer 12, the version being compared (is it M1a vs M1b or M1a vs M2a?) and the mean relatadness given by humans (hint: the repo you cloned will come useful here, both in terms of code and data). Print the head of the dataframe to check everything is in order, and check also that the number of stimuli match with your number across the assignment (starting from task 1).

In [ ]:
# your code here

## Task 4.2: Correlate with human judgements and visualise [8 points]

First, correlate the cosine similarities at the different layers to the mean human relatedness judgements. Use the same correlation metric used by Trott & Bergen.

In [ ]:
# your code here

Next, visualise your results. You want to see the correlation between BERT embeddings and human judgements per layer, but what would also be interesting is to include the meaning contrasts (such as M1_a_M1_b, etc), so that we can see how those play out per layer.

In [ ]:
# your code here

### Reflect on the correlations and on the visualisations. What can you observe and infer in terms of which layer(s) might be capturing meaning best? Is there one way to determine that (i.e., what does 'capturing meanings' mean?)? Contrast and compare the layers. [5 points]

%your answer here



